# 저장된 ML 모델 기반 Mock IoT 안전 알림 데모

앞 노트북이 저장한 가공 데이터, 학습된 모델, 임계값을 불러옵니다. 이 노트북에서는 데이터 전처리·합성 라벨 생성·모델 재학습을 하지 않고, 12개의 새로운 Mock IoT 입력에 대한 시연만 수행합니다.

- 정상: 체크리스트 없음
- 주의·경고: 저장된 ML 위험 점수와 임계값으로 판단
- 긴급: 설비별 명시적 비상조건이 ML 판단보다 우선


In [ ]:
from pathlib import Path
import json
import warnings

import joblib
import pandas as pd

warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', 60)
pd.set_option('display.max_colwidth', 120)
pd.set_option('display.float_format', lambda x: f'{x:,.4f}')

PROCESSED_PATH = Path('data/processed_industrial_fire_ml.csv')
MODEL_PATH = Path('models/industrial_fire_accident_pipeline.joblib')
METADATA_PATH = Path('models/industrial_fire_accident_metadata.json')

required_files = [PROCESSED_PATH, MODEL_PATH, METADATA_PATH]
missing_files = [str(path) for path in required_files if not path.exists()]
if missing_files:
    raise FileNotFoundError(
        '먼저 industrial_fire_custom_accident_ml.ipynb를 끝까지 실행하세요. '
        f'없는 파일: {missing_files}'
    )


## 1. 가공 데이터·학습 모델·임계값 불러오기


In [ ]:
processed_df = pd.read_csv(PROCESSED_PATH, parse_dates=['timestamp'])
model = joblib.load(MODEL_PATH)
metadata = json.loads(METADATA_PATH.read_text(encoding='utf-8'))

feature_columns = metadata['feature_columns']
warning_threshold = float(metadata['warning_threshold'])
caution_threshold = float(metadata['caution_threshold'])

missing_columns = set(feature_columns + [metadata['target_column']]) - set(processed_df.columns)
if missing_columns:
    raise ValueError(f'가공 데이터에 필수 컬럼이 없습니다: {sorted(missing_columns)}')

print(f'가공 데이터: {len(processed_df):,}행 × {processed_df.shape[1]}열')
print(f"저장 모델: {metadata['model_name']}")
print(f'주의 임계값: {caution_threshold:.3f}')
print(f'경고 임계값: {warning_threshold:.3f}')
print(f"테스트 PR-AUC: {metadata['test_metrics']['pr_auc']:.4f}")
display(processed_df.head(3))


## 2. 새로운 Mock IoT 입력 12건 예측

Mock 값은 이미 각 설비에서 측정된 센서값으로 간주하므로 다시 보정하지 않습니다.


In [ ]:
mock_rows = [
 # 4개 기계의 정상 상태
 {'event_id':'EVT-001','machine_id':'M-0101','machine_type':'REACTOR','Factory':'Factory_A','Region':'Urban','Shift':'Day','Workers':8,'Exp':'Senior','Training':'Yes','Temp':29,'Pressure':20,'Humidity':58,'Vibration':0.8,'Speed':1100,'Age':2,'Service_Days':15,'Gas':1.5,'Sparks':0},
 {'event_id':'EVT-002','machine_id':'M-0102','machine_type':'COMPRESSOR','Factory':'Factory_A','Region':'Urban','Shift':'Day','Workers':10,'Exp':'Senior','Training':'Yes','Temp':24,'Pressure':21,'Humidity':55,'Vibration':1.4,'Speed':1800,'Age':3,'Service_Days':25,'Gas':1.5,'Sparks':0},
 {'event_id':'EVT-003','machine_id':'M-0103','machine_type':'STORAGE_TANK','Factory':'Factory_A','Region':'Rural','Shift':'Day','Workers':9,'Exp':'Senior','Training':'Yes','Temp':20,'Pressure':18,'Humidity':60,'Vibration':0.5,'Speed':800,'Age':2,'Service_Days':12,'Gas':2.0,'Sparks':0},
 {'event_id':'EVT-004','machine_id':'M-0104','machine_type':'PUMP','Factory':'Factory_A','Region':'Urban','Shift':'Day','Workers':11,'Exp':'Senior','Training':'Yes','Temp':24,'Pressure':18,'Humidity':54,'Vibration':1.3,'Speed':1700,'Age':4,'Service_Days':30,'Gas':1.2,'Sparks':0},
 # 주의 후보
 {'event_id':'EVT-005','machine_id':'M-0101','machine_type':'REACTOR','Factory':'Factory_A','Region':'Urban','Shift':'Night','Workers':20,'Exp':'Junior','Training':'No','Temp':35,'Pressure':25,'Humidity':48,'Vibration':1.6,'Speed':1500,'Age':8,'Service_Days':130,'Gas':4.5,'Sparks':1},
 {'event_id':'EVT-006','machine_id':'M-0104','machine_type':'PUMP','Factory':'Factory_A','Region':'Industrial_Zone','Shift':'Night','Workers':22,'Exp':'Junior','Training':'No','Temp':28,'Pressure':17,'Humidity':47,'Vibration':3.2,'Speed':2600,'Age':10,'Service_Days':180,'Gas':2.0,'Sparks':0},
 # 기계별 경고 후보
 {'event_id':'EVT-007','machine_id':'M-0101','machine_type':'REACTOR','Factory':'Factory_A','Region':'Industrial_Zone','Shift':'Night','Workers':35,'Exp':'Junior','Training':'No','Temp':40,'Pressure':36,'Humidity':40,'Vibration':2.0,'Speed':1800,'Age':13,'Service_Days':250,'Gas':6.5,'Sparks':2},
 {'event_id':'EVT-008','machine_id':'M-0102','machine_type':'COMPRESSOR','Factory':'Factory_A','Region':'Industrial_Zone','Shift':'Night','Workers':38,'Exp':'Junior','Training':'No','Temp':34,'Pressure':35,'Humidity':38,'Vibration':4.8,'Speed':3900,'Age':15,'Service_Days':285,'Gas':4.0,'Sparks':1},
 {'event_id':'EVT-009','machine_id':'M-0103','machine_type':'STORAGE_TANK','Factory':'Factory_A','Region':'Industrial_Zone','Shift':'Night','Workers':40,'Exp':'Junior','Training':'No','Temp':32,'Pressure':28,'Humidity':31,'Vibration':1.5,'Speed':1300,'Age':14,'Service_Days':270,'Gas':8.5,'Sparks':2},
 {'event_id':'EVT-010','machine_id':'M-0104','machine_type':'PUMP','Factory':'Factory_A','Region':'Industrial_Zone','Shift':'Night','Workers':36,'Exp':'Junior','Training':'No','Temp':33,'Pressure':14,'Humidity':40,'Vibration':4.6,'Speed':3300,'Age':16,'Service_Days':300,'Gas':3.0,'Sparks':1},
 # 긴급 안전조건 확인
 {'event_id':'EVT-011','machine_id':'M-0101','machine_type':'REACTOR','Factory':'Factory_A','Region':'Industrial_Zone','Shift':'Night','Workers':42,'Exp':'Junior','Training':'No','Temp':45,'Pressure':42,'Humidity':35,'Vibration':2.5,'Speed':2000,'Age':17,'Service_Days':330,'Gas':8.0,'Sparks':3},
 {'event_id':'EVT-012','machine_id':'M-0103','machine_type':'STORAGE_TANK','Factory':'Factory_A','Region':'Industrial_Zone','Shift':'Night','Workers':44,'Exp':'Junior','Training':'No','Temp':38,'Pressure':36,'Humidity':28,'Vibration':1.8,'Speed':1500,'Age':18,'Service_Days':345,'Gas':9.6,'Sparks':4},
]
mock_df = pd.DataFrame(mock_rows)
missing_features = set(feature_columns) - set(mock_df.columns)
if missing_features:
    raise ValueError(f'Mock 입력에 필수 컬럼이 없습니다: {sorted(missing_features)}')
mock_df['ml_accident_probability'] = model.predict_proba(mock_df[feature_columns])[:, 1]

def emergency_reason(row):
    reasons = []
    if row.machine_type == 'REACTOR':
        if row.Temp >= 42: reasons.append('반응기 비상 온도 조건')
        if row.Pressure >= 45: reasons.append('반응기 비상 압력 조건')
        if row.Gas >= 8.5 and row.Sparks >= 2: reasons.append('가스·스파크 복합 비상 조건')
    elif row.machine_type == 'COMPRESSOR':
        if row.Vibration >= 4.8: reasons.append('압축기 비상 진동 조건')
        if row.Speed >= 3900: reasons.append('압축기 비상 회전속도 조건')
        if row.Pressure >= 45: reasons.append('압축기 비상 압력 조건')
    elif row.machine_type == 'STORAGE_TANK':
        if row.Gas >= 9 and row.Sparks >= 3: reasons.append('가스·스파크 복합 비상 조건')
        if row.Pressure >= 45: reasons.append('탱크 비상 압력 조건')
    elif row.machine_type == 'PUMP':
        if row.Vibration >= 5.5: reasons.append('펌프 비상 진동 조건')
        if row.Pressure <= 10 and row.Vibration >= 4.5: reasons.append('저압·고진동 복합 비상 조건')
    return ', '.join(reasons)

mock_df['emergency_reason'] = mock_df.apply(emergency_reason, axis=1)
def decide_status(row):
    if row.emergency_reason: return '긴급'
    if row.ml_accident_probability >= warning_threshold: return '경고'
    if row.ml_accident_probability >= caution_threshold: return '주의'
    return '정상'
mock_df['status'] = mock_df.apply(decide_status, axis=1)
mock_df['risk_percent'] = (mock_df['ml_accident_probability'] * 100).round(1)
display(mock_df[['event_id','machine_id','machine_type','Temp','Pressure','Vibration','Speed','Gas','Sparks','risk_percent','status','emergency_reason']])

## 3. 비정상 상태에만 설비별 체크리스트 출력


In [ ]:
manual_ids = metadata['manual_ids']
actions = {
 'REACTOR': {
  '주의':['온도·압력 값을 재확인한다.','최근 1시간 추세를 확인한다.','이상 내용을 기록한다.'],
  '경고':['설비 관리자에게 즉시 보고한다.','반응기 주변 접근을 제한한다.','냉각·압력 제어 계통 점검을 요청한다.'],
  '긴급':['현장 비상대응 절차를 시행한다.','위험구역을 통제하고 인원을 대피시킨다.','승인 전까지 재가동하지 않는다.']},
 'COMPRESSOR': {
  '주의':['압력·진동·속도를 재확인한다.','이상 소음을 안전한 위치에서 확인한다.','정비 이력을 확인한다.'],
  '경고':['설비 관리자에게 즉시 보고한다.','압축기 주변 접근을 제한한다.','베어링·윤활·축 정렬 점검을 요청한다.'],
  '긴급':['비상대응 절차를 시행한다.','회전체와 연결부 접근을 금지한다.','정비 담당자 확인 전 재가동하지 않는다.']},
 'STORAGE_TANK': {
  '주의':['가스·온도·압력 값을 재확인한다.','스파크 감지 횟수를 확인한다.','이상 상태를 기록한다.'],
  '경고':['주변 작업자에게 경고를 전달한다.','점화 가능 작업을 중단한다.','가스 센서와 환기 장치 점검을 요청한다.'],
  '긴급':['현장 비상대응 절차를 시행한다.','점화원을 통제하고 위험구역에서 대피한다.','탱크를 임의로 조작하지 않는다.']},
 'PUMP': {
  '주의':['흡입·토출 압력과 진동을 재확인한다.','이상 소음과 누출 흔적을 확인한다.','최근 정비 이력을 확인한다.'],
  '경고':['설비 관리자에게 즉시 보고한다.','펌프 주변 접근을 제한한다.','씰·베어링·윤활·배관 연결부 점검을 요청한다.'],
  '긴급':['현장 비상대응 절차를 시행한다.','회전체와 누출 의심부 접근을 금지한다.','정비 담당자 확인 전 재가동하지 않는다.']},
}
mock_df['manual_id'] = mock_df['machine_type'].map(manual_ids)
mock_df['checklist'] = mock_df.apply(lambda row: [] if row.status == '정상' else actions[row.machine_type][row.status], axis=1)

for _, row in mock_df.iterrows():
    print('='*72)
    print(f"[{row.status}] {row.event_id} | {row.machine_id} | ML 위험도 {row.risk_percent:.1f}%")
    if row.status == '정상':
        print('체크리스트 없음')
        continue
    if row.emergency_reason: print(f'긴급 근거: {row.emergency_reason}')
    print(f'참조 매뉴얼: {row.manual_id}')
    for i, action in enumerate(row.checklist, 1): print(f'  [ ] {i}. {action}')

assert mock_df.loc[mock_df.status.eq('정상'), 'checklist'].map(len).eq(0).all()
assert {'정상','주의','경고','긴급'}.issubset(set(mock_df['status']))
status_summary = mock_df.status.value_counts().reindex(['정상','주의','경고','긴급'], fill_value=0).to_frame('count')
display(status_summary)